In [18]:
import torch
import torchvision
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F

transform = transforms.Compose(
    [transforms.Resize([300,300]), # resize the image to its stock size of 300x300
     transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

testset = datasets.ImageFolder('testdata', transform=transform) 
testloader = torch.utils.data.DataLoader(testset, batch_size=30,
                                          shuffle=True)

classes = ('cherry', 'strawberry', 'tomato')

import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5) # take our 3 channel data and change to 6 channel with a kernal size of 5
        self.pool = nn.MaxPool2d(2, 2) # giving a pooling size of 2,2 means that our image size is halved
        self.conv2 = nn.Conv2d(6, 16, 5) # take our 6 channel data and change it to 16 channel data with same kernal size 5
        # based on kernal size, stride, padding and image size we get the below calculation
        self.fc1 = nn.Linear(16 * 72 * 72, 120) 
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) 
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net() 
PATH = 'model.pth'
net.load_state_dict(torch.load(PATH))

<All keys matched successfully>

In [19]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 1500 test images: {100 * correct // total} %')

Accuracy of the network on the 1500 test images: 71 %


In [20]:
# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# again no gradients needed
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1


# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

Accuracy for class: cherry is 69.5 %
Accuracy for class: strawberry is 59.3 %
Accuracy for class: tomato is 85.0 %
